# 01 · Exploración y validación de los datos. Este notebook sirve para **conocer los archivos** antes de graficar y para **verificar** que los números del dashboard son correctos.
Requisito: con el entorno virtual activo, ejecute una vez `pip install ipykernel` y elija como kernel el Python de `.venv`.

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # permite importar src/
import pandas as pd
from src import config, analisis
from src.carga_datos import cargar_mortalidad, cargar_divipola, cargar_codigos_muerte, integrar_datos

## 1. Carga de cada archivo por separado

In [2]:
mortalidad = cargar_mortalidad()
divipola = cargar_divipola()
causas = cargar_codigos_muerte()
print(mortalidad.shape, divipola.shape, causas.shape)
mortalidad.head()

(244355, 16) (1123, 4) (12568, 4)


,COD_DANE,COD_DEPARTAMENTO,COD_MUNICIPIO,AREA_DEFUNCION,SITIO_DEFUNCION,ANO,MES,HORA,MINUTOS,SEXO,ESTADO_CIVIL,GRUPO_EDAD1,NIVEL_EDUCATIVO,MANERA_MUERTE,COD_MUERTE,IDPROFESIONAL
0,66682,66,682,1,5,2019,11,23,0,2,5.0,15,3,Homicidio,X994,1
1,17001,17,1,1,1,2019,12,21,10,1,6.0,16,2,Natural,C712,1
2,25843,25,843,1,1,2019,12,20,15,2,4.0,19,2,Sin Determinar,Y349,1
3,23001,23,1,1,1,2019,12,14,20,2,1.0,23,2,Natural,I678,1
4,73001,73,1,1,1,2019,11,8,40,2,4.0,23,2,Natural,F03X,1


## 2. Estructura y valores faltantes

In [3]:
mortalidad.info()
mortalidad.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 244355 entries, 0 to 244354
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   COD_DANE          244355 non-null  string 
 1   COD_DEPARTAMENTO  244355 non-null  string 
 2   COD_MUNICIPIO     244355 non-null  int64  
 3   AREA_DEFUNCION    244355 non-null  int64  
 4   SITIO_DEFUNCION   244355 non-null  int64  
 5   ANO               244355 non-null  int64  
 6   MES               244355 non-null  Int64  
 7   HORA              244355 non-null  int64  
 8   MINUTOS           244355 non-null  int64  
 9   SEXO              244355 non-null  Int64  
 10  ESTADO_CIVIL      244354 non-null  float64
 11  GRUPO_EDAD1       244355 non-null  Int64  
 12  NIVEL_EDUCATIVO   244355 non-null  int64  
 13  MANERA_MUERTE     244355 non-null  string 
 14  COD_MUERTE        244355 non-null  string 
 15  IDPROFESIONAL     244355 non-null  int64  
dtypes: Int64(3), float64(1), int64(

COD_DANE            0
COD_DEPARTAMENTO    0
COD_MUNICIPIO       0
AREA_DEFUNCION      0
SITIO_DEFUNCION     0
ANO                 0
MES                 0
HORA                0
MINUTOS             0
SEXO                0
ESTADO_CIVIL        1
GRUPO_EDAD1         0
NIVEL_EDUCATIVO     0
MANERA_MUERTE       0
COD_MUERTE          0
IDPROFESIONAL       0
dtype: int64

In [4]:
# Valores posibles de las variables codificadas
for col in ['MES', 'SEXO', 'GRUPO_EDAD1']:
    print(col, sorted(mortalidad[col].dropna().unique()))

MES [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
SEXO [np.int64(1), np.int64(2), np.int64(3)]
GRUPO_EDAD1 [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29)]


## 3. Calidad de los cruces (joins)

In [5]:
sin_mpio = ~mortalidad['COD_DANE'].isin(divipola['COD_DANE'])
sin_causa = ~mortalidad['COD_MUERTE'].isin(causas['COD_MUERTE'])
print('Muertes sin municipio en DIVIPOLA:', sin_mpio.sum())
print('Muertes sin causa en CIE-10 (4 car.):', sin_causa.sum())
mortalidad.loc[sin_causa, 'COD_MUERTE'].value_counts().head(10)

Muertes sin municipio en DIVIPOLA: 0
Muertes sin causa en CIE-10 (4 car.): 0


Series([], Name: count, dtype: Int64)

## 4. Integración y comprobaciones

In [6]:
df = integrar_datos(mortalidad, divipola, causas)
assert len(df) == len(mortalidad), 'El merge duplicó o perdió filas'
print('Total de muertes:', len(df))

Total de muertes: 244355


In [7]:
# Las sumas de cada agregación deben coincidir con el total
assert analisis.muertes_por_mes(df)['TOTAL'].sum() == len(df)
assert analisis.muertes_por_departamento(df)['TOTAL'].sum() == len(df)
assert analisis.distribucion_grupo_edad(df)['TOTAL'].sum() == len(df)
assert analisis.muertes_sexo_departamento(df)['TOTAL'].sum() == len(df)
print('Todas las sumas cuadran')

Todas las sumas cuadran


## 5. Tablas de cada visualización

In [8]:
analisis.top_causas(df)

,POSICION,COD_MUERTE,CAUSA,TOTAL,PORCENTAJE
1085,1,I219,"Infarto agudo del miocardio, sin otra especifi...",35088,14.36
1354,2,J449,"Enfermedad pulmonar obstructiva cronica, no es...",7210,2.95
1351,3,J440,Enfermedad pulmonar obstructiva cronica con in...,6445,2.64
1321,4,J189,"Neumonia, no especificada",5798,2.37
273,5,C169,"Tumor maligno del estomago, parte no especificada",5125,2.10
334,6,C349,"Tumor maligno de los bronquios o del pulmon, p...",4438,1.82
3033,7,X954,"Agresion con disparo de otras armas de fuego, ...",4396,1.80
401,8,C509,"Tumor maligno de la mama, parte no especificada",3619,1.48
421,9,C61X,Tumor maligno de la prostata,3437,1.41
1065,10,I10X,Hipertension esencial (primaria),3317,1.36


In [9]:
analisis.top_ciudades_homicidios(df)

,COD_DANE,MUNICIPIO,DEPARTAMENTO,TOTAL,CIUDAD
718,76001,Santiago de Cali,Valle del Cauca,1239,Santiago de Cali (Valle del Cauca)
132,11001,"Bogotá, D.C.","Bogotá, D.C.",1148,"Bogotá, D.C. (Bogotá, D.C.)"
0,05001,Medellín,Antioquia,637,Medellín (Antioquia)
113,08001,Barranquilla,Atlántico,345,Barranquilla (Atlántico)
561,54001,San José de Cúcuta,Norte de Santander,262,San José de Cúcuta (Norte de Santander)


In [10]:
analisis.ciudades_menor_mortalidad(df, 20)  # ver los empates más allá del 10.º lugar

,COD_DANE,MUNICIPIO,DEPARTAMENTO,TOTAL,CIUDAD
573,27025,Alto Baudó,Chocó,1,Alto Baudó (Chocó)
463,25095,Bituima,Cundinamarca,1,Bituima (Cundinamarca)
690,50245,El Calvario,Meta,1,El Calvario (Meta)
1085,91263,El Encanto,Amazonas,1,El Encanto (Amazonas)
883,68344,Hato,Santander,1,Hato (Santander)
743,52390,La Tola,Nariño,1,La Tola (Nariño)
1095,94663,Mapiripana,Guainía,1,Mapiripana (Guainía)
167,13440,Margarita,Bolívar,1,Margarita (Bolívar)
592,27495,Nuquí,Chocó,1,Nuquí (Chocó)
1088,91530,Puerto Alegría,Amazonas,1,Puerto Alegría (Amazonas)


In [11]:
analisis.distribucion_grupo_edad(df)

,GRUPO_EDAD,TOTAL,RANGO,CODIGOS,PORCENTAJE
0,Mortalidad neonatal,4520,Menor de 1 mes,0–4,1.85
1,Mortalidad infantil,2771,1 a 11 meses,5–6,1.13
2,Primera infancia,1518,1 a 4 años,7–8,0.62
3,Niñez,1993,5 a 14 años,9–10,0.82
4,Adolescencia,3795,15 a 19 años,11,1.55
5,Juventud,11840,20 a 29 años,12–13,4.85
6,Adultez temprana,17276,30 a 44 años,14–16,7.07
7,Adultez intermedia,29105,45 a 59 años,17–19,11.91
8,Vejez,115453,60 a 84 años,20–24,47.25
9,Longevidad / Centenarios,56061,85 a 100+ años,25–28,22.94


In [12]:
# Comprobación manual de la transformación de GRUPO_EDAD1
pd.crosstab(df['GRUPO_EDAD1'], df['GRUPO_EDAD'])

GRUPO_EDAD,Adolescencia,Adultez intermedia,Adultez temprana,Edad desconocida,Juventud,Longevidad / Centenarios,Mortalidad infantil,Mortalidad neonatal,Niñez,Primera infancia,Vejez
GRUPO_EDAD1,,,,,,,,,,,
0,0,0,0,0,0,0,0,313,0,0,0
1,0,0,0,0,0,0,0,971,0,0,0
2,0,0,0,0,0,0,0,1743,0,0,0
3,0,0,0,0,0,0,0,1425,0,0,0
4,0,0,0,0,0,0,0,68,0,0,0
5,0,0,0,0,0,0,1847,0,0,0,0
6,0,0,0,0,0,0,924,0,0,0,0
7,0,0,0,0,0,0,0,0,0,747,0
8,0,0,0,0,0,0,0,0,0,771,0


6. Definición de homicidio
Compara la definición por código CIE-10 (X85–Y09) con la columna MANERA_MUERTE del DANE.

In [13]:
es_x85_y09 = mortalidad['COD_MUERTE'].str[:3].isin(config.PREFIJOS_HOMICIDIO)
pd.crosstab(es_x85_y09, mortalidad['MANERA_MUERTE'])

MANERA_MUERTE,Accidente,Estudio,Homicidio,Natural,Sin Determinar,Suicidio
COD_MUERTE,,,,,,
False,12332,978,34,212703,2335,2930
True,0,0,13043,0,0,0
